# Persona 2: NLP, clasificación y validación

Este cuaderno utiliza **dataset_con_ocr.xlsx**, entregado por la persona 1. Sube el Excel en el punto 1 y ejecuta las celdas de arriba hacia abajo. Se aprovecha el OCR existente; no se vuelve a procesar el ZIP. Las métricas se calculan al ejecutar el cuaderno. Revisa las etiquetas y los grupos de documentos relacionados antes de interpretar los resultados.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils
!pip -q install pytesseract pdf2image wordcloud jiwer nltk seaborn

In [ ]:
import os, re, zipfile, unicodedata, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageOps, ImageFilter
import pytesseract
from pdf2image import convert_from_path, pdfinfo_from_path
from IPython.display import display
from google.colab import files
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from wordcloud import WordCloud
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
SALIDA=Path('/content/resultados_persona2'); SALIDA.mkdir(exist_ok=True)
print('Listo.')

## 1. Cargar el Excel con OCR
Pulsa ▶ y selecciona **dataset_con_ocr.xlsx**. El cuaderno valida las columnas y reporta los textos vacíos. No es necesario cargar el ZIP para el análisis NLP.

In [ ]:
subidos=files.upload()
nombre=next((k for k in subidos if k.lower().endswith('.xlsx')),None)
if not nombre: raise ValueError('Selecciona dataset_con_ocr.xlsx y ejecuta otra vez.')
datos=pd.read_excel(nombre,engine='openpyxl')
datos.columns=[str(c).strip().lower() for c in datos.columns]
columnas={'carpeta','archivo','institucion','etiqueta','formato','texto_ocr'}
if not columnas.issubset(datos.columns): raise ValueError('Faltan columnas: '+str(columnas-set(datos.columns)))
datos=datos.dropna(subset=['archivo']).copy()
for col in ['archivo','carpeta','institucion','formato','texto_ocr']:
    datos[col]=datos[col].fillna('').astype(str)
datos['etiqueta']=pd.to_numeric(datos['etiqueta'],errors='coerce')
if not datos.etiqueta.isin([0,1]).all(): raise ValueError('Las etiquetas deben ser 0 o 1.')
if datos.duplicated(['carpeta','archivo']).any(): raise ValueError('Hay filas duplicadas por carpeta y archivo.')
print('Registros:',len(datos)); print('Distribución por clase:'); display(datos.etiqueta.value_counts())
display(datos[['archivo','institucion','etiqueta','formato']].head())

## 2. Revisar el OCR proporcionado
El texto proviene del Excel de la persona 1. Los documentos sin texto se muestran y se excluyen del análisis; para incluirlos, pide que se reprocesen y vuelve a cargar el Excel corregido. La comparación objetiva de OCR requiere transcripciones de referencia y se realiza en el punto 5.

In [ ]:
df=datos.copy()
df['grupo']=df.archivo.map(lambda a:'serie_143492' if a.startswith('143492-') else a)
df['error']=''
df['paginas']='No indicado'
# Conservar formato original; se desconoce cuántas páginas procesó el OCR.
df.to_csv(SALIDA/'dataset_ocr_utilizado.csv',index=False,encoding='utf-8-sig')
print('Registros sin texto OCR:',df.texto_ocr.str.strip().eq('').sum())
display(df.loc[df.texto_ocr.str.strip().eq(''),['archivo','institucion','etiqueta']])
display(df[['archivo','institucion','etiqueta','formato','grupo']].head())

**Revisión de grupos:** Las imágenes `143492` se mantienen juntas por precaución. Asigna el mismo grupo a otras páginas, copias o versiones del mismo comunicado para evitar que aparezcan simultáneamente en entrenamiento y prueba.

In [ ]:
# Para cambiar grupos, descomenta y adapta los nombres:
# df.loc[df.archivo.isin(['archivo1.png','archivo2.png']), 'grupo'] = 'mismo_documento'
print('Grupos listos para revisión.')

## 3. Limpieza y técnicas NLP
Se conservan las tildes. El resumen es extractivo: selecciona oraciones representativas del propio texto, por lo que puede heredar errores del OCR.

In [ ]:
stop=set(stopwords.words('spanish')) | {'ser','asi','ademas','tambien','peru','perú'}
def normalizar(s):
    s=unicodedata.normalize('NFKC',str(s)).lower()
    s=re.sub(r'[^a-záéíóúüñ\s]', ' ',s)
    return ' '.join(w for w in s.split() if w not in stop and len(w)>2)
df['texto_limpio']=df.texto_ocr.fillna('').map(normalizar)
validos=df[(df.error=='') & (df.texto_limpio.str.strip()!='')].copy().reset_index(drop=True)
print('Documentos válidos:',len(validos)); display(validos[['archivo','texto_limpio']].head())
if validos.empty: raise ValueError('No hay textos válidos. Revisa la sección OCR.')

In [ ]:
texto_total=' '.join(validos.texto_limpio)
wc=WordCloud(width=1600,height=800,background_color='white',colormap='Blues',collocations=False).generate(texto_total)
plt.figure(figsize=(14,7)); plt.imshow(wc); plt.axis('off'); plt.title('Palabras frecuentes en el conjunto'); plt.tight_layout(); plt.savefig(SALIDA/'nube_palabras.png',dpi=200); plt.show()
vector=TfidfVectorizer(ngram_range=(1,2),min_df=1,max_features=5000)
X=vector.fit_transform(validos.texto_limpio)
palabras=pd.DataFrame({'termino':vector.get_feature_names_out(),'tfidf_promedio':np.asarray(X.mean(axis=0)).ravel()}).sort_values('tfidf_promedio',ascending=False).head(20)
display(palabras); palabras.to_csv(SALIDA/'palabras_clave.csv',index=False,encoding='utf-8-sig')
plt.figure(figsize=(9,7)); sns.barplot(data=palabras,x='tfidf_promedio',y='termino',color='#2775a5'); plt.title('20 términos destacados por TF-IDF promedio'); plt.tight_layout(); plt.savefig(SALIDA/'palabras_clave.png',dpi=200); plt.show()

In [ ]:
def resumen_extractivo(texto, n=3):
    oraciones=[re.sub(r'\s+',' ',x).strip() for x in re.split(r'(?<=[.!?])\s+|\n+',str(texto)) if len(x.strip())>=40]
    if len(oraciones)<=n: return ' '.join(oraciones) if oraciones else str(texto)[:800]
    limpias=[normalizar(x) for x in oraciones]
    if not any(limpias): return ' '.join(oraciones[:n])
    try:
        mat=TfidfVectorizer().fit_transform(limpias)
        scores=np.asarray(mat.sum(axis=1)).ravel()
        indices=sorted(np.argsort(scores)[-n:])
        return ' '.join(oraciones[i] for i in indices)
    except ValueError: return ' '.join(oraciones[:n])
validos['resumen']=validos.texto_ocr.map(resumen_extractivo)
validos[['archivo','etiqueta','resumen']].to_csv(SALIDA/'resumenes.csv',index=False,encoding='utf-8-sig')
for _,fila in validos.head(3).iterrows(): print('\nARCHIVO:',fila.archivo,'\nRESUMEN:',fila.resumen[:1500])

## 4. Clasificación y validación cruzada
Se usa TF-IDF + regresión logística con 5 particiones estratificadas por grupo. Cada documento se predice solo cuando estuvo fuera del entrenamiento; el TF-IDF se ajusta dentro de cada partición para evitar fuga de información. La clase positiva es **El Niño = 1**. Los grupos deben revisarse manualmente antes de ejecutar.

In [ ]:
print('Distribución de documentos:'); display(validos.etiqueta.value_counts().rename(index={0:'No El Niño',1:'El Niño'}))
print('Grupos por clase:'); display(validos.groupby('etiqueta').grupo.nunique())
print('Grupos con archivos múltiples:'); display(validos.groupby('grupo').filter(lambda x: len(x)>1)[['archivo','grupo','etiqueta']])
if (validos.groupby('grupo').etiqueta.nunique()>1).any(): raise ValueError('Un grupo tiene etiquetas contradictorias; revísalo.')
if validos.groupby('etiqueta').grupo.nunique().min()<5: raise ValueError('Se necesitan al menos 5 grupos independientes por clase para esta validación.')
modelo=Pipeline([('tfidf',TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True)),('clasificador',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42))])
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
splits=list(cv.split(validos.texto_limpio,validos.etiqueta,validos.grupo))
for i,(tr,te) in enumerate(splits,1):
    if len(set(validos.etiqueta.iloc[tr]))<2 or len(set(validos.etiqueta.iloc[te]))<2: raise ValueError(f'La partición {i} no tiene ambas clases. Revisar grupos o reducir n_splits.')
pred=cross_val_predict(modelo,validos.texto_limpio,validos.etiqueta,cv=splits)
reporte=classification_report(validos.etiqueta,pred,target_names=['No El Niño','El Niño'],output_dict=True,zero_division=0)
print(classification_report(validos.etiqueta,pred,target_names=['No El Niño','El Niño'],zero_division=0))
pd.DataFrame(reporte).T.to_csv(SALIDA/'metricas_clasificador.csv',encoding='utf-8-sig')
validos['prediccion_cv']=pred
validos[['archivo','grupo','etiqueta','prediccion_cv']].to_csv(SALIDA/'predicciones_validacion.csv',index=False,encoding='utf-8-sig')
cm=confusion_matrix(validos.etiqueta,pred,labels=[0,1])
plt.figure(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['No El Niño','El Niño'],yticklabels=['No El Niño','El Niño']); plt.xlabel('Predicción'); plt.ylabel('Etiqueta real'); plt.title('Matriz de confusión: validación cruzada'); plt.tight_layout(); plt.savefig(SALIDA/'matriz_confusion.png',dpi=200); plt.show()
r=reporte['El Niño']; plt.figure(figsize=(7,4)); sns.barplot(x=['Precisión','Recall','F1'],y=[r['precision'],r['recall'],r['f1-score']],color='#2775a5'); plt.ylim(0,1); plt.ylabel('Valor'); plt.title('Desempeño para la clase El Niño'); plt.tight_layout(); plt.savefig(SALIDA/'metricas_clase_nino.png',dpi=200); plt.show()

**Interpretación:** precisión = proporción de predicciones El Niño que son correctas; recall = proporción de comunicados El Niño detectados; F1 = equilibrio entre ambas. Con pocos documentos y clases desequilibradas, estas cifras son exploratorias. En este ZIP las clases también están asociadas al formato y a la institución, por lo que el modelo podría aprender esas diferencias en lugar del tema. No es evidencia de rendimiento en instituciones o formatos nuevos. Revisar especialmente etiquetas y fuentes cercanas.

## 4.1 Entrenamiento final y guardado de los modelos
Después de evaluar con validación cruzada por grupos, esta sección vuelve a utilizar la misma configuración: **TF-IDF + regresión logística**. Entrena un modelo final con todos los textos válidos para usarlo con nuevos documentos, y además guarda los cinco modelos de las particiones. Las métricas que se reportan siguen siendo las de validación cruzada, no las obtenidas con el modelo final sobre sus propios datos.

El archivo `modelo_clasificador.joblib` incluye el vectorizador TF-IDF ajustado y el clasificador. El ZIP también incluye la función de limpieza y sus palabras vacías, para que el modelo procese el texto nuevo de la misma manera. Conserva estos archivos juntos.

In [ ]:
import joblib, json, sklearn, platform
from sklearn.base import clone

# Guardar los cinco modelos que corresponden a la evaluación por grupos.
carpeta_modelos = SALIDA / 'modelos_validacion_cruzada'
carpeta_modelos.mkdir(exist_ok=True)
for numero, (indices_entrenamiento, indices_prueba) in enumerate(splits, start=1):
    modelo_pliegue = clone(modelo)
    modelo_pliegue.fit(validos.texto_limpio.iloc[indices_entrenamiento], validos.etiqueta.iloc[indices_entrenamiento])
    joblib.dump(modelo_pliegue, carpeta_modelos / f'modelo_pliegue_{numero}.joblib')

# Modelo de uso: mismo pipeline, ajustado con todos los textos válidos.
modelo_final = clone(modelo)
modelo_final.fit(validos.texto_limpio, validos.etiqueta)
ruta_modelo = SALIDA / 'modelo_clasificador.joblib'
joblib.dump(modelo_final, ruta_modelo)

# Exportar las palabras eliminadas y una función reutilizable de limpieza.
(SALIDA / 'stopwords_espanol.json').write_text(json.dumps(sorted(stop), ensure_ascii=False, indent=2), encoding='utf-8')
codigo_prediccion = r"""import json, re, unicodedata
from pathlib import Path
import joblib

BASE = Path(__file__).resolve().parent
with open(BASE / 'stopwords_espanol.json', encoding='utf-8') as archivo:
    STOPWORDS = set(json.load(archivo))

def limpiar_texto(texto):
    texto = unicodedata.normalize('NFKC', str(texto)).lower()
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    return ' '.join(palabra for palabra in texto.split() if palabra not in STOPWORDS and len(palabra) > 2)

def predecir(texto):
    limpio = limpiar_texto(texto)
    if not limpio:
        raise ValueError('El texto está vacío después de la limpieza.')
    modelo = joblib.load(BASE / 'modelo_clasificador.joblib')
    clase = int(modelo.predict([limpio])[0])
    return {'etiqueta': clase, 'categoria': 'El Niño' if clase == 1 else 'No El Niño'}
"""
(SALIDA / 'predecir_comunicado.py').write_text(codigo_prediccion, encoding='utf-8')

info = {
    'descripcion': 'Pipeline TF-IDF + regresión logística',
    'modelo_final': 'Entrenado con todos los textos válidos tras validación cruzada por grupos',
    'validacion': 'StratifiedGroupKFold; 5 particiones; shuffle=True; random_state=42',
    'documentos_entrenamiento_final': len(validos),
    'documentos_por_clase': {str(int(k)): int(v) for k,v in validos.etiqueta.value_counts().items()},
    'etiquetas': {'0': 'No El Niño', '1': 'El Niño'},
    'version_python': platform.python_version(),
    'version_scikit_learn': sklearn.__version__,
    'version_joblib': joblib.__version__,
    'entrada_del_pipeline': 'texto previamente limpiado; para texto OCR sin limpiar usar predecir_comunicado.py',
    'metricas': 'metricas_clasificador.csv contiene las predicciones fuera de entrenamiento de validación cruzada',
}
(SALIDA / 'informacion_modelo.json').write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')

# Comprobar que el archivo guardado se carga y produce la misma predicción.
modelo_recuperado = joblib.load(ruta_modelo)
assert np.array_equal(modelo_final.predict(validos.texto_limpio), modelo_recuperado.predict(validos.texto_limpio))
print('Modelo final guardado:', ruta_modelo)
print('Modelos de validación guardados:', len(splits))
print('Archivos para usar con textos nuevos: modelo_clasificador.joblib, predecir_comunicado.py y stopwords_espanol.json')


### Ejemplo para clasificar un comunicado nuevo
Ejecuta esta celda después de entrenar. El texto de ejemplo es solo una demostración. Para usarlo en otra aplicación, conserva juntos `predecir_comunicado.py`, `stopwords_espanol.json` y `modelo_clasificador.joblib`, e instala versiones compatibles de scikit-learn y joblib. Carga únicamente archivos joblib de confianza.

In [ ]:
import sys
if str(SALIDA) not in sys.path: sys.path.insert(0, str(SALIDA))
from predecir_comunicado import predecir
texto_ejemplo = 'La entidad publica recomendaciones ante el fenómeno El Niño.'
print('Texto:', texto_ejemplo)
print('Clasificación:', predecir(texto_ejemplo))

## 5. Calidad del OCR: CER y WER
Para una medición real se necesita texto de referencia correcto **del mismo documento y de las mismas páginas**. Crearás una plantilla CSV, completarás `texto_referencia` mediante transcripción humana o texto digital verificado y clasificarás `tipo_fuente` como `pdf_digital` o `imagen_redes`. **No se asume que todo PDF sea digital ni que toda imagen proceda de redes.** Usa varios ejemplos de cada grupo, documenta cómo se eligieron y verifica que sean comparables. No rellenes referencias con el resultado OCR.

CER = errores de caracteres / caracteres de referencia; WER = errores de palabras / palabras de referencia. Más bajo es mejor. Puede superar 100 % por inserciones.

In [ ]:
plantilla=validos[['archivo','formato','paginas','texto_ocr']].copy()
plantilla['tipo_fuente']=''; plantilla['texto_referencia']=''
plantilla.to_csv(SALIDA/'plantilla_referencias_ocr.csv',index=False,encoding='utf-8-sig')
files.download(str(SALIDA/'plantilla_referencias_ocr.csv'))
print('Descarga, completa y guarda la plantilla como CSV UTF-8. Después ejecuta la siguiente celda.')

In [ ]:
from jiwer import cer, wer
print('Sube la plantilla completada. Si todavía no tienes referencias, puedes omitir esta celda.')
subidos_ref=files.upload()
nombre_ref=next((k for k in subidos_ref if k.lower().endswith('.csv')),None)
if not nombre_ref: raise ValueError('Sube la plantilla CSV completada.')
ref=pd.read_csv(nombre_ref,keep_default_na=False)
requeridas={'archivo','tipo_fuente','texto_referencia','texto_ocr'}
if not requeridas.issubset(ref.columns): raise ValueError('Faltan columnas: '+str(requeridas-set(ref.columns)))
ref=ref[ref.tipo_fuente.isin(['pdf_digital','imagen_redes']) & ref.texto_referencia.str.strip().ne('')].copy()
if ref.empty: raise ValueError('No hay referencias completadas con tipo_fuente válido.')
def comparable(s): return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',str(s)).lower()).strip()
ref['CER']=ref.apply(lambda r:cer(comparable(r.texto_referencia),comparable(r.texto_ocr)),axis=1)
ref['WER']=ref.apply(lambda r:wer(comparable(r.texto_referencia),comparable(r.texto_ocr)),axis=1)
display(ref[['archivo','tipo_fuente','CER','WER']])
res=ref.groupby('tipo_fuente').agg(documentos=('archivo','count'),CER_promedio=('CER','mean'),WER_promedio=('WER','mean'))
display(res); ref.to_csv(SALIDA/'calidad_ocr_por_documento.csv',index=False,encoding='utf-8-sig'); res.to_csv(SALIDA/'calidad_ocr_por_tipo.csv',encoding='utf-8-sig')
ax=res[['CER_promedio','WER_promedio']].plot.bar(figsize=(8,5),rot=0); ax.set_ylabel('Tasa de error (menor es mejor)'); ax.set_title('Calidad OCR por tipo de fuente'); plt.tight_layout(); plt.savefig(SALIDA/'calidad_ocr.png',dpi=200); plt.show()

## 6. Texto base para el informe
El siguiente texto se completa con las métricas obtenidas. Antes de entregarlo revisa las etiquetas, los grupos y la selección de referencias. La calidad del resumen se describe cualitativamente, no se afirma una precisión que no se haya medido.

In [ ]:
p,r,f,_=precision_recall_fscore_support(validos.etiqueta,pred,labels=[1],zero_division=0)
texto=f"""PRUEBAS Y VALIDACIÓN
Se analizaron {len(validos)} archivos con texto OCR válido: {sum(validos.etiqueta==1)} etiquetados como relacionados con El Niño y {sum(validos.etiqueta==0)} como no relacionados. Se utilizó el texto OCR proporcionado por la persona 1 en el archivo Excel; el método y las páginas procesadas deben confirmarse con ella. Las etiquetas se tomaron del Excel y requieren revisión manual del contenido.

Para el análisis NLP se aplicó normalización, eliminación de palabras frecuentes, nube de palabras, extracción de términos mediante TF-IDF y resumen extractivo. El resumen selecciona oraciones existentes; su utilidad debe revisarse manualmente, así como posibles errores heredados del OCR.

Se evaluó un clasificador TF-IDF con regresión logística mediante validación cruzada estratificada de 5 particiones por grupo. Para la clase El Niño, la precisión fue {p[0]:.2%}, el recall {r[0]:.2%} y el F1 {f[0]:.2%}. Las predicciones y la matriz de confusión se incluyen en los archivos de resultados.

La calidad del OCR se evalúa mediante CER y WER comparando cada extracción con una transcripción correcta de las mismas páginas. Los resultados por tipo de fuente solo pueden reportarse cuando se completen las referencias. No se debe interpretar el formato como medida de calidad.

Limitaciones: tamaño reducido, desequilibrio entre clases, posible similitud entre documentos y asociación entre institución, formato y clase. Los resultados son exploratorios y podrían no generalizar a fuentes nuevas. Se recomienda ampliar ambas clases, revisar etiquetas y grupos, equilibrar formatos y evaluar documentos de instituciones no vistas."""
print(texto)
(SALIDA/'texto_pruebas_y_validacion.txt').write_text(texto,encoding='utf-8')

## 7. Descargar todos los resultados
El ZIP incluye `modelo_clasificador.joblib`, los cinco modelos de validación, la función para predecir textos nuevos, las palabras de limpieza, la información del modelo, las gráficas y las métricas. Si completaste la medición OCR, sus resultados también se incluirán. Guarda el notebook en tu Drive desde **Archivo → Guardar una copia en Drive**.

In [ ]:
import shutil
zip_salida=shutil.make_archive('/content/resultados_persona2','zip',SALIDA)
files.download(zip_salida)
print('Descarga lista:',zip_salida)